# Non-Maximum Suppression (NMS)

Wiki reference for [the NMS algorithm](https://ml-viz-ruby.vercel.app/wiki/nms-algorithm).

**The idea in one sentence.** Object detectors emit many overlapping boxes per object; NMS keeps
the highest-scoring box and **suppresses** every other box whose **IoU** with it exceeds a
threshold — a simple greedy dedup whose one knob (the IoU threshold) trades **duplicate removal**
against **suppressing genuinely-overlapping objects** in crowds.

We implement IoU and NMS from scratch, **validate IoU and that NMS removes duplicates**, then
cover the gotchas.

> **To save your work:** click **Copy to Drive**, or File → Save a copy in Drive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

plt.style.use('dark_background')

## From-scratch IoU and NMS

In [ ]:
def iou(box_a, box_b):
    """IoU for boxes in [x1, y1, x2, y2] format."""
    xa = max(box_a[0], box_b[0]); ya = max(box_a[1], box_b[1])
    xb = min(box_a[2], box_b[2]); yb = min(box_a[3], box_b[3])
    inter = max(0, xb - xa) * max(0, yb - ya)
    area_a = (box_a[2] - box_a[0]) * (box_a[3] - box_a[1])
    area_b = (box_b[2] - box_b[0]) * (box_b[3] - box_b[1])
    union = area_a + area_b - inter
    return inter / union if union > 0 else 0.0

def nms(boxes, scores, iou_threshold=0.5, score_threshold=0.0):
    """
    Standard greedy NMS.
    boxes:  (N, 4) [x1, y1, x2, y2]
    scores: (N,)
    Returns: kept indices (original ordering)
    """
    mask = scores > score_threshold
    idxs  = np.where(mask)[0]
    order = idxs[np.argsort(scores[idxs])[::-1]]
    kept  = []
    while len(order) > 0:
        best = order[0]
        kept.append(best)
        remaining = [j for j in order[1:]
                     if iou(boxes[best], boxes[j]) <= iou_threshold]
        order = np.array(remaining)
    return kept

def soft_nms(boxes, scores, sigma=0.5, score_threshold=0.01):
    """
    Soft-NMS: Gaussian score decay instead of hard removal.
    Returns: kept indices (score above threshold after decay)
    """
    scores = scores.copy().astype(float)
    n = len(boxes)
    indices = list(range(n))
    kept = []

    while len(indices) > 0:
        # Find index with max score
        best_local = int(np.argmax(scores[indices]))
        best = indices[best_local]
        if scores[best] < score_threshold:
            break
        kept.append(best)
        indices.pop(best_local)
        # Decay scores of remaining boxes
        for j in indices:
            ov = iou(boxes[best], boxes[j])
            scores[j] *= np.exp(-ov**2 / sigma)

    return kept

### Validate: IoU behaves correctly

Intersection-over-Union is 1 for identical boxes, 0 for disjoint boxes, and in between for
partial overlap. We confirm the extremes and a half-overlap case in between.

In [ ]:
half = iou([0, 0, 10, 10], [5, 0, 15, 10])   # boxes overlap on half their area
print(f'IoU(identical)={iou([0,0,10,10],[0,0,10,10]):.2f}, IoU(disjoint)={iou([0,0,10,10],[20,20,30,30]):.2f}, IoU(half overlap)={half:.3f}')
assert iou([0,0,10,10],[0,0,10,10]) == 1.0, 'IoU of identical boxes is 1'
assert iou([0,0,10,10],[20,20,30,30]) == 0.0, 'IoU of disjoint boxes is 0'
assert 0 < half < 1, 'partial overlap gives an IoU strictly between 0 and 1'
print('\n✅ IoU measures overlap in [0,1] — the similarity NMS thresholds on')

## Reproduce the worked trace

In [ ]:
# Four boxes around a single cat; see worked trace in the wiki page
# Boxes designed so IoU(0,1) ≈ 0.75 (suppress) and IoU(2,3) = 0.25 (keep)
boxes_trace = np.array([
    [10, 10, 50, 50],   # score 0.92  — IoU with box1 ≈ 0.75 → box1 suppressed
    [13, 13, 53, 53],   # score 0.87  — heavy overlap with box 0
    [60, 10, 100, 50],  # score 0.61  — IoU with box3 = 0.25 → box3 kept
    [84, 10, 124, 50],  # score 0.55  — light overlap with box 2
], dtype=float)
scores_trace = np.array([0.92, 0.87, 0.61, 0.55])

print("Pairwise IoU matrix:")
n = len(boxes_trace)
for i in range(n):
    for j in range(i+1, n):
        print(f"  IoU(box_{i} s={scores_trace[i]}, box_{j} s={scores_trace[j]}) = "
              f"{iou(boxes_trace[i], boxes_trace[j]):.2f}")

kept = nms(boxes_trace, scores_trace, iou_threshold=0.5)
suppressed = [i for i in range(n) if i not in kept]
print(f"\nNMS result:")
print(f"  Kept (indices):      {kept}  → scores {scores_trace[kept]}")
print(f"  Suppressed (indices): {suppressed}  → scores {scores_trace[suppressed]}")

### Validate: NMS removes the duplicate detection

Run on four boxes around one cat, NMS keeps the highest-scoring box and suppresses its
near-duplicate (IoU ~0.75 > 0.5), while a genuinely separate box (IoU 0.25) survives. We confirm
the top box is kept and its duplicate is dropped.

In [ ]:
print(f'kept indices: {sorted(kept)}  (scores {scores_trace[sorted(kept)]})')
assert 0 in kept, 'the highest-scoring box is always kept'
assert 1 not in kept, 'its near-duplicate (IoU > threshold) is suppressed'
print('\n✅ NMS greedily keeps the best box and drops its overlapping duplicates')

## Visualise: kept vs suppressed boxes

In [ ]:
def draw_boxes(ax, boxes, scores, kept_set, title):
    for i, (b, s) in enumerate(zip(boxes, scores)):
        x1, y1, x2, y2 = b
        is_kept = i in kept_set
        color = '#6366f1' if is_kept else '#ef4444'
        style = '-' if is_kept else '--'
        lw    = 2.5 if is_kept else 1.5
        rect = patches.Rectangle(
            (x1, y1), x2-x1, y2-y1,
            linewidth=lw, edgecolor=color,
            facecolor=color, alpha=0.15 if is_kept else 0.07,
            linestyle=style
        )
        ax.add_patch(rect)
        label = f"s={s:.2f}\n{'KEEP' if is_kept else 'SUPPRESS'}"
        ax.text(x1+2, y1+4, label, color=color, fontsize=8)
    ax.set_xlim(0, 120); ax.set_ylim(0, 80)
    ax.set_title(title); ax.set_aspect('equal')
    ax.grid(True, alpha=0.1)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
all_idxs = list(range(len(boxes_trace)))

draw_boxes(axes[0], boxes_trace, scores_trace, all_idxs, 'All candidate boxes')
draw_boxes(axes[1], boxes_trace, scores_trace, set(kept), 'After NMS (IoU threshold 0.5)')

plt.suptitle('NMS: removing duplicate detections', y=1.01, fontsize=13)
plt.tight_layout()
plt.show()

## Soft-NMS vs hard NMS on a crowded scene

In [ ]:
# Simulate 3 people standing close together
# Each person generates 2 overlapping candidate boxes
people_boxes = np.array([
    # Person 1 (left)
    [10, 5, 40, 70],  [12, 6, 42, 72],
    # Person 2 (middle, overlapping person 1)
    [30, 5, 60, 70],  [31, 6, 61, 71],
    # Person 3 (right, overlapping person 2)
    [50, 5, 80, 70],  [52, 6, 82, 72],
], dtype=float)
people_scores = np.array([0.95, 0.88, 0.91, 0.82, 0.89, 0.76])

kept_hard = nms(people_boxes, people_scores, iou_threshold=0.5)
kept_soft = soft_nms(people_boxes, people_scores, sigma=0.5, score_threshold=0.3)

print(f"Hard NMS kept {len(kept_hard)} box(es): indices {kept_hard}")
print(f"Soft-NMS kept {len(kept_soft)} box(es): indices {kept_soft}")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
draw_boxes(axes[0], people_boxes, people_scores, set(kept_hard), f'Hard NMS (kept {len(kept_hard)})')
draw_boxes(axes[1], people_boxes, people_scores, set(kept_soft), f'Soft-NMS (kept {len(kept_soft)})')
plt.suptitle('Crowded scene: Soft-NMS preserves more distinct detections', y=1.01, fontsize=12)
plt.tight_layout()
plt.show()

## Gotchas & tradeoffs

| Gotcha | Consequence |
|--------|-------------|
| **threshold choice** | strict misses crowded objects, loose keeps duplicates (demo) |
| **hard suppression** | deleting a box outright loses a valid overlapping detection — Soft-NMS |
| **class-agnostic vs per-class** | usually run NMS per class |
| **greedy** | a lower-scored but better-localized box can be suppressed |
| **speed** | naive NMS is $O(n^2)$; sort + vectorize for many boxes |

Demo: the IoU threshold trades duplicate removal against suppressing real crowded objects.

In [ ]:
# The NMS threshold is a genuine trade-off in CROWDED scenes. A LOW IoU threshold suppresses
# aggressively (good for duplicates, but it can also delete a real object that legitimately
# overlaps a neighbour); a HIGH threshold keeps more boxes (fewer missed objects, but more
# duplicates survive). We sweep it on 6 boxes covering 3 close-standing people.
n_strict = len(nms(people_boxes, people_scores, iou_threshold=0.3))
n_loose = len(nms(people_boxes, people_scores, iou_threshold=0.9))
print(f'boxes kept: strict threshold (0.3) -> {n_strict},  loose threshold (0.9) -> {n_loose}  (6 candidates, 3 people)')
assert n_strict < n_loose, 'a strict (low) IoU threshold suppresses more; a loose (high) one keeps duplicates'
print('\nNo single threshold is right for every scene -> crowds need higher thresholds or Soft-NMS.')

## ✏️ Your turn

**Exercise 1:** Implement **vectorised IoU**: given a single box `q` and an array of N boxes, return an array of N IoU values using NumPy broadcasting (no Python loop).

**Exercise 2:** Implement **class-aware NMS**: given boxes, scores, and class labels, run NMS independently per class and merge the results.

**Exercise 3:** How does the number of kept boxes change as you sweep the IoU threshold from 0.1 to 0.9 on the 6-box crowded scene? Plot the curve.

In [ ]:
# Exercise 1: vectorised IoU
def iou_vectorised(query_box, boxes):
    """
    query_box: (4,)  [x1, y1, x2, y2]
    boxes:     (N, 4)
    Returns:   (N,)  IoU values
    """
    # TODO(you): use np.maximum and broadcasting
    pass

# Test
# ious = iou_vectorised(boxes_trace[0], boxes_trace)
# print(ious)  # first entry should be 1.0 (self-overlap)

In [ ]:
# Exercise 3: threshold sweep
thresholds = np.linspace(0.1, 0.9, 17)
n_kept = []
for tau in thresholds:
    # TODO(you): run nms with each threshold on people_boxes/people_scores
    pass

# plt.plot(thresholds, n_kept, 'o-', color='#6366f1')
# plt.xlabel('IoU threshold'); plt.ylabel('Boxes kept')
# plt.title('Effect of NMS threshold on crowded scene')
# plt.grid(True, alpha=0.2)
# plt.show()

<details>
<summary>Solution — Exercise 1</summary>

```python
def iou_vectorised(query_box, boxes):
    xa = np.maximum(query_box[0], boxes[:, 0])
    ya = np.maximum(query_box[1], boxes[:, 1])
    xb = np.minimum(query_box[2], boxes[:, 2])
    yb = np.minimum(query_box[3], boxes[:, 3])
    inter = np.maximum(0, xb - xa) * np.maximum(0, yb - ya)
    area_q = (query_box[2]-query_box[0]) * (query_box[3]-query_box[1])
    area_b = (boxes[:,2]-boxes[:,0]) * (boxes[:,3]-boxes[:,1])
    union  = area_q + area_b - inter
    return np.where(union > 0, inter / union, 0.0)
```
</details>

## Key takeaways

- **NMS = greedy dedup:** keep the top-scored box, suppress boxes with IoU above a threshold.
- **IoU** measures overlap in $[0,1]$ (verified).
- **It removes duplicate detections** while keeping separate objects (verified).
- **The IoU threshold is a trade-off:** strict over-suppresses crowds, loose keeps duplicates
  (demo) — Soft-NMS softens this.